# Vanguard A/B Test — Análisis de Resultados
**Juan Boberg Aguirre · Ironhack Data Analytics Bootcamp**

## Contexto
Vanguard realizó un experimento A/B entre el 15 de marzo y el 20 de junio de 2017 para evaluar si un rediseño de su proceso de onboarding mejoraba la tasa de finalización.

- **Control:** proceso original
- **Test:** nuevo diseño con UI mejorada y prompts contextuales
- **Objetivo:** aumentar la completion rate en al menos un 5%

## Hipótesis
- **H1:** El nuevo diseño consigue que más usuarios terminen el proceso
- **H2:** El nuevo diseño hace el proceso más rápido
- **H3:** El nuevo diseño reduce los errores y retrocesos

## 1. Carga de datos
Cargamos los tres datasets necesarios:
- `df_final_web_data_pt_1` y `pt_2`: rastro de interacciones digitales (se unen en un solo DataFrame)
- `df_final_demo`: perfil demográfico de los clientes
- `df_final_experiment_clients`: asignación de cada cliente a Control o Test

Los archivos están en la carpeta `data/` del repositorio.

In [1]:
import pandas as pd
import numpy as np

STEP_ORDER = ["start", "step_1", "step_2", "step_3", "confirm"]

# Cargamos los datos de interacciones web (dos partes) desde la carpeta data/
df_pt1 = pd.read_csv("data/df_final_web_data_pt_1.txt")
df_pt2 = pd.read_csv("data/df_final_web_data_pt_2.txt")

# Cargamos el perfil demográfico de clientes
df_demo = pd.read_csv("data/df_final_demo.txt")

# Cargamos la asignación de clientes a grupos (Control/Test)
df_experiment = pd.read_csv("data/df_final_experiment_clients.txt")

print("Web data pt1:", df_pt1.shape)
print("Web data pt2:", df_pt2.shape)
print("Demo:", df_demo.shape)
print("Experiment:", df_experiment.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'data/df_final_web_data_pt_1.txt'

## 2. Unión y preparación de datos
Concatenamos las dos partes del dataset web y lo unimos con la asignación de grupos mediante un merge por `client_id`.

In [ ]:
# Unimos las dos partes del dataset web
df = pd.concat([df_pt1, df_pt2], ignore_index=True)

# Añadimos la columna Variation (Control/Test) haciendo merge con df_experiment
df = df.merge(df_experiment[["client_id", "Variation"]], on="client_id", how="left")

print("Dataset combinado:", df.shape)
print(df["Variation"].value_counts())
print(df.head())

## 3. Consolidación de pasos por usuario
Creamos una función que transforma el dataset de eventos (una fila por interacción) en un dataset de usuarios (una fila por cliente), indicando qué pasos completó cada uno y hasta dónde llegó.

In [ ]:
def consolidate_steps(df, client_col="client_id", step_col="process_step"):
    """
    Transforma el dataset de eventos en uno de usuarios.
    Para cada cliente indica qué pasos completó (0/1) y cuál fue el último paso alcanzado.
    """
    dummies = pd.get_dummies(df[[client_col, step_col]], columns=[step_col])
    step_cols = [f"{step_col}_{s}" for s in STEP_ORDER if f"{step_col}_{s}" in dummies.columns]
    result = dummies.groupby(client_col)[step_cols].max().reset_index()
    result.columns = [client_col] + [c.replace(f"{step_col}_", "") for c in step_cols]
    df_cat = df.copy()
    df_cat[step_col] = pd.Categorical(df_cat[step_col], categories=STEP_ORDER, ordered=True)
    last_step = df_cat.groupby(client_col)[step_col].max().rename("last_step")
    result = result.merge(last_step, on=client_col)
    return result.reset_index(drop=True)

# Separamos los grupos y consolidamos
df_control = df[df["Variation"] == "Control"].drop(columns="Variation")
df_test = df[df["Variation"] == "Test"].drop(columns="Variation")

result_control = consolidate_steps(df_control)
result_test = consolidate_steps(df_test)

# Contamos cuántas veces visitó cada usuario cada paso (para el error rate)
step_counts_control = (
    df_control.groupby(["client_id", "process_step"])
    .size().unstack(fill_value=0)
    .reindex(columns=STEP_ORDER, fill_value=0).reset_index()
)
step_counts_test = (
    df_test.groupby(["client_id", "process_step"])
    .size().unstack(fill_value=0)
    .reindex(columns=STEP_ORDER, fill_value=0).reset_index()
)

print("Control:", result_control.shape, "| Test:", result_test.shape)

## 4. KPIs
### KPI 1 — Completion Rate
Porcentaje de usuarios que llegaron al paso `confirm` sobre el total de usuarios del grupo.
Este es el KPI principal del experimento.

In [ ]:
completion_rate_control = (result_control["last_step"] == "confirm").mean() * 100
completion_rate_test = (result_test["last_step"] == "confirm").mean() * 100

print(f"Completion rate (control): {completion_rate_control:.2f}%")
print(f"Completion rate (test):    {completion_rate_test:.2f}%")
print(f"Diferencia:                +{completion_rate_test - completion_rate_control:.2f}pp")

### KPI 2 — Time per Step
Mediana de segundos que los usuarios pasan en cada paso del proceso.
Calculamos el tiempo como la diferencia entre el timestamp de un paso y el siguiente del mismo usuario.

In [ ]:
def time_per_step(df_group):
    """Calcula la mediana de tiempo en segundos por paso del proceso."""
    df_sorted = df_group.copy()
    df_sorted["date_time"] = pd.to_datetime(df_sorted["date_time"])
    df_sorted = df_sorted.sort_values(["client_id", "date_time"])
    df_sorted["time_next"] = df_sorted.groupby("client_id")["date_time"].shift(-1)
    df_sorted["time_spent"] = (df_sorted["time_next"] - df_sorted["date_time"]).dt.total_seconds()
    return df_sorted.groupby("process_step")["time_spent"].median().reindex(STEP_ORDER).reset_index()

tps_control = time_per_step(df_control)
tps_control.columns = ["paso", "control_seg"]
tps_test = time_per_step(df_test)
tps_test.columns = ["paso", "test_seg"]
tps = tps_control.merge(tps_test, on="paso")
tps["diferencia_seg"] = (tps["test_seg"] - tps["control_seg"]).round(1)
print(tps)

### KPI 3 — Error Rate
Porcentaje de usuarios que completaron el proceso pero visitaron al menos un paso más de una vez (retrocedieron o se equivocaron).
Solo se calcula sobre los usuarios que llegaron a `confirm`.

In [ ]:
def error_rate(result_group, step_counts_group):
    """Calcula el % de usuarios que completaron pero cometieron al menos un error (retroceso)."""
    completed = result_group[result_group["last_step"] == "confirm"][["client_id"]]
    completed_counts = completed.merge(step_counts_group, on="client_id")
    completed_counts["tuvo_error"] = (completed_counts[STEP_ORDER] > 1).any(axis=1)
    return completed_counts["tuvo_error"].mean() * 100

er_control = error_rate(result_control, step_counts_control)
er_test = error_rate(result_test, step_counts_test)

print(f"Error rate (control): {er_control:.2f}%")
print(f"Error rate (test):    {er_test:.2f}%")
print(f"Diferencia:           {er_test - er_control:.2f}pp")

### KPI 4 — Momento de Abandono
Distribución del abandono entre los usuarios que NO completaron el proceso.
Indica en qué paso se perdió cada usuario.

In [ ]:
def abandonment(result_group):
    """Calcula la distribución del abandono por paso entre usuarios que no completaron."""
    abandoned = result_group[result_group["last_step"] != "confirm"]
    ab = abandoned["last_step"].value_counts().reindex(STEP_ORDER[:-1]).reset_index()
    ab.columns = ["paso_abandono", "clientes"]
    ab["pct_abandono_%"] = (ab["clientes"] / len(abandoned) * 100).round(2)
    return ab

ab_control = abandonment(result_control)
ab_control.columns = ["paso_abandono", "clientes_control", "pct_control_%"]
ab_test = abandonment(result_test)
ab_test.columns = ["paso_abandono", "clientes_test", "pct_test_%"]
ab = ab_control.merge(ab_test, on="paso_abandono")
ab["diferencia_pp"] = (ab["pct_test_%"] - ab["pct_control_%"]).round(2)
print(ab)

## 5. Resumen del análisis de KPIs

**H1 ✅ Confirmada** — El nuevo diseño mejora la completion rate en +3.71pp (65.59% → 69.29%). La mejora es real y consistente.

**H2 ✅ Confirmada** — El proceso es 24 segundos más rápido en mediana con el nuevo diseño, especialmente en step_3 (-16s). El tiempo alto en confirm (+311s) es llamativo pero no compensa la mejora global.

**H3 ❌ No confirmada** — La tasa de error es prácticamente igual en ambos grupos (53.42% vs 55.10%). El rediseño no tocó los puntos donde la gente se lía.

**Hallazgo clave:** El nuevo diseño reduce el abandono en start (-10.94pp) pero ese abandono se redistribuye hacia step_1 (+5.94pp) y step_3 (+3.57pp). Engancha mejor al principio pero no resuelve la fricción posterior.

## 6. Test de Hipótesis
### H1 — Completion Rate
Usamos un Z-test de proporciones para determinar si la diferencia en completion rate entre grupos es estadísticamente significativa o podría deberse al azar.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

n_control = result_control.shape[0]
n_test = result_test.shape[0]
completados_control = (result_control["last_step"] == "confirm").sum()
completados_test = (result_test["last_step"] == "confirm").sum()

print(f"Control: {completados_control}/{n_control} ({completados_control/n_control*100:.2f}%)")
print(f"Test:    {completados_test}/{n_test} ({completados_test/n_test*100:.2f}%)")

counts = np.array([completados_test, completados_control])
nobs = np.array([n_test, n_control])
stat, p_value = proportions_ztest(counts, nobs)

print(f"\nZ-statistic: {stat:.4f}")
print(f"P-value:     {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\n✅ Diferencia estadísticamente significativa (p < {alpha})")
else:
    print(f"\n❌ Diferencia NO estadísticamente significativa (p >= {alpha})")

In [ ]:
# Verificamos si supera el umbral mínimo del 5%
diferencia = (completados_test/n_test) - (completados_control/n_control)
umbral = 0.05

print(f"Diferencia observada: {diferencia*100:.2f}pp")
print(f"Umbral requerido:     {umbral*100:.2f}pp")

if diferencia >= umbral:
    print(f"\n✅ La mejora supera el umbral del 5%")
else:
    print(f"\n❌ La mejora NO supera el umbral del 5% — requiere iteración")

**Conclusión H1:** La diferencia es estadísticamente significativa (p ≈ 0.0000) pero no supera el umbral mínimo del 5%. El resultado no es fruto del azar pero tampoco es suficiente para recomendar la implementación inmediata.

### H2 adicional — Abandono en Start
Verificamos si la reducción del abandono en el primer paso es estadísticamente significativa.

In [ ]:
abandono_start_control = (result_control["last_step"] == "start").sum()
abandono_start_test = (result_test["last_step"] == "start").sum()

print(f"Control: {abandono_start_control}/{n_control} ({abandono_start_control/n_control*100:.2f}%)")
print(f"Test:    {abandono_start_test}/{n_test} ({abandono_start_test/n_test*100:.2f}%)")

counts = np.array([abandono_start_test, abandono_start_control])
nobs = np.array([n_test, n_control])
stat, p_value = proportions_ztest(counts, nobs)

print(f"\nZ-statistic: {stat:.4f}")
print(f"P-value:     {p_value:.4f}")

if p_value < 0.05:
    print("\n✅ Diferencia estadísticamente significativa")

**Conclusión:** La reducción del abandono en start (-4.86pp) es estadísticamente significativa y muy robusta (Z = -17.17). Confirma que el nuevo diseño mejora notablemente la entrada al flujo.

## 7. Evaluación del Experimento

### Efectividad del diseño

| Aspecto | Resultado |
|---|---|
| Objetivo principal (CR > +5%) | ❌ No cumplido (+3.71pp) |
| Significancia estadística | ✅ Confirmada (p < 0.05) |
| Mejora en entrada al flujo | ✅ Sí (-10.94pp en abandono de start) |
| Reducción de errores | ❌ No (diferencia mínima: +1.68pp) |
| Velocidad del proceso | ✅ Mejora leve (-24s de mediana total) |
| **Recomendación** | **No implementar — requiere iteración** |

### Duración del experimento

In [ ]:
# Verificamos el rango de fechas del experimento
df["date_time"] = pd.to_datetime(df["date_time"])

print("Fecha inicio:", df["date_time"].min())
print("Fecha fin:   ", df["date_time"].max())
print("Duración:    ", (df["date_time"].max() - df["date_time"].min()).days, "días")

print("\nControl:")
print("  Inicio:", df[df["Variation"]=="Control"]["date_time"].min())
print("  Fin:   ", df[df["Variation"]=="Control"]["date_time"].max())
print("\nTest:")
print("  Inicio:", df[df["Variation"]=="Test"]["date_time"].min())
print("  Fin:   ", df[df["Variation"]=="Test"]["date_time"].max())

**Valoración de la duración:** 97 días es adecuado — captura múltiples ciclos de comportamiento y el tamaño muestral (50.500 usuarios) permite detectar diferencias pequeñas con alta potencia estadística. Riesgo: posible efecto de novedad en los usuarios del Test al inicio del experimento.

### Datos adicionales necesarios
- **Comportamiento dentro del paso:** clics, scroll depth y campos que generan abandonos
- **Dispositivo y canal:** móvil vs escritorio, canal de entrada
- **Segmentación:** primera vez en el proceso vs usuario recurrente
- **Datos post-proceso:** satisfacción, retención e incidencias tras completar

## 8. Conclusión final
El nuevo diseño tiene impacto real (+3.71pp en completion rate, estadísticamente significativo) pero no suficiente para justificar su implementación inmediata. La recomendación es iterar atacando específicamente step_1 y step_3, donde se concentra el abandono redistribuido, y reducir la fricción en el paso confirm antes de lanzar un nuevo test A/B.